# Project 1 - Part 3: Deep Learning

This notebook contains the image classification project development process using deep learning algorithms.

## Project Goals:
- Prepare a 3-class image dataset (cup, pen, keyboard)
- Create Convolutional Neural Network (CNN) model
- Perform model training and evaluation
- Apply transfer learning
- Optimize and improve model performance

## Dataset Structure:
```
proje_veri_seti/
├── bardak/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
├── kalem/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
└── klavye/
    ├── 1.jpg
    ├── 2.jpg
    └── ...
```

In [ ]:
# Loading required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Sklearn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Others
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 8)

# TensorFlow GPU check
print("TensorFlow version:", tf.__version__)
print("GPU availability:", tf.config.list_physical_devices('GPU'))
print("CUDA compatibility:", tf.test.is_built_with_cuda())

## 1. Dataset Preparation and Loading

In [ ]:
# Dataset folder paths
data_dir = 'proje_veri_seti'
classes = ['bardak', 'kalem', 'klavye']

# Check if dataset folder exists
if not os.path.exists(data_dir):
    print(f"Dataset folder '{data_dir}' not found!")
    print("Please follow these steps:")
    print("1. Create 'proje_veri_seti' folder")
    print("2. Create subfolders for each class (bardak, kalem, klavye)")
    print("3. Add at least 20-50 images to each folder")
    print("4. Save images in .jpg, .jpeg or .png format")
else:
    print(f"Dataset folder '{data_dir}' found!")

# Dataset statistics
def count_images_in_dataset(data_dir, classes):
    """Count the number of images in the dataset"""
    stats = {}
    total_images = 0
    
    for class_name in classes:
        class_path = os.path.join(data_dir, class_name)
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            count = len(image_files)
            stats[class_name] = count
            total_images += count
            print(f"{class_name}: {count} images")
        else:
            stats[class_name] = 0
            print(f"{class_name}: Folder not found!")
    
    print(f"\nTotal images: {total_images}")
    return stats, total_images

# Count images
if os.path.exists(data_dir):
    image_stats, total_count = count_images_in_dataset(data_dir, classes)
    
    if total_count > 0:
        print("\n✅ Dataset is ready for training!")
    else:
        print("\n⚠️ No images found in dataset folders!")
else:
    print("\n❌ Dataset folder does not exist!")

In [ ]:
# Demo için örnek görüntüler oluşturma (Eğer veri seti yoksa)
def create_sample_dataset():
    """Demo amaçlı örnek görüntüler oluştur"""
    print("Demo amaçlı örnek veri seti oluşturuluyor...")
    
    # Her sınıf için klasör oluştur
    for class_name in classes:
        class_path = os.path.join(data_dir, class_name)
        os.makedirs(class_path, exist_ok=True)
        
        # Her sınıf için 10 örnek görüntü oluştur
        for i in range(1, 11):
            # Rastgele renkli görüntü oluştur
            img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            
            # Sınıfa özgü basit şekiller ekle
            if class_name == 'bardak':
                # Silindir şekli (bardak benzeri)
                cv2.rectangle(img, (80, 50), (144, 180), (100, 150, 200), -1)
                cv2.ellipse(img, (112, 50), (32, 16), 0, 0, 360, (120, 170, 220), -1)
            elif class_name == 'kalem':
                # Uzun ince şekil (kalem benzeri)
                cv2.rectangle(img, (100, 30), (124, 190), (150, 100, 50), -1)
                cv2.circle(img, (112, 30), 12, (200, 150, 100), -1)
            elif class_name == 'klavye':
                # Dikdörtgen şekil (klavye benzeri)
                cv2.rectangle(img, (50, 80), (174, 140), (80, 80, 80), -1)
                # Tuşlar
                for x in range(60, 165, 15):
                    for y in range(90, 131, 15):
                        cv2.rectangle(img, (x, y), (x+10, y+10), (120, 120, 120), -1)
            
            # Görüntüyü kaydet
            img_path = os.path.join(class_path, f'{i}.jpg')
            cv2.imwrite(img_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    
    print("Demo veri seti oluşturuldu!")
    return count_images_in_dataset(data_dir, classes)

# Eğer veri seti boşsa, demo veri seti oluştur
if os.path.exists(data_dir):
    _, total = count_images_in_dataset(data_dir, classes)
    if total == 0:
        create_sample_dataset()
else:
    os.makedirs(data_dir, exist_ok=True)
    create_sample_dataset()

In [ ]:
# Veri setinden örnek görüntüleri görselleştir
def display_sample_images(data_dir, classes, samples_per_class=3):
    """Her sınıftan örnek görüntüleri göster"""
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(15, 5*len(classes)))
    
    for i, class_name in enumerate(classes):
        class_path = os.path.join(data_dir, class_name)
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            for j in range(min(samples_per_class, len(image_files))):
                img_path = os.path.join(class_path, image_files[j])
                img = Image.open(img_path)
                
                if len(classes) == 1:
                    axes[j].imshow(img)
                    axes[j].set_title(f'{class_name} - {image_files[j]}')
                    axes[j].axis('off')
                else:
                    axes[i, j].imshow(img)
                    axes[i, j].set_title(f'{class_name} - {image_files[j]}')
                    axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.show()

# Örnek görüntüleri göster
if os.path.exists(data_dir):
    display_sample_images(data_dir, classes)

## 2. Data Preprocessing and Data Augmentation

In [ ]:
# Image parameters
img_height, img_width = 224, 224
batch_size = 32
num_classes = len(classes)

# Data Augmentation settings
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # 20% reserved for validation
)

# Test data - only normalization
test_datagen = ImageDataGenerator(rescale=1./255)

# Training dataset
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Validation dataset
validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Class indices: {train_generator.class_indices}")

In [ ]:
# Visualize data augmentation examples
def show_augmented_images():
    """Show data augmentation examples"""
    # Get a batch of data
    images, labels = next(train_generator)
    
    # Show first 8 images
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    
    class_names = list(train_generator.class_indices.keys())
    
    for i in range(8):
        if i < len(images):
            axes[i].imshow(images[i])
            # Find class name from label
            class_idx = np.argmax(labels[i])
            axes[i].set_title(f'Class: {class_names[class_idx]}')
            axes[i].axis('off')
    
    plt.suptitle('Data Augmentation Examples', fontsize=16)
    plt.tight_layout()
    plt.show()

# Show data augmentation examples
if train_generator.samples > 0:
    show_augmented_images()
    # Reset generator
    train_generator.reset()

## 3. Creating Convolutional Neural Network (CNN) Model

In [ ]:
# Creating simple CNN model
def create_simple_cnn(input_shape, num_classes):
    """Create simple CNN model"""
    model = keras.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Fourth Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create model
input_shape = (img_height, img_width, 3)
model = create_simple_cnn(input_shape, num_classes)

# Show model summary
model.summary()

# Visualize model architecture
tf.keras.utils.plot_model(
    model, 
    to_file='model_architecture.png', 
    show_shapes=True, 
    show_layer_names=True,
    rankdir='TB',
    dpi=150
)

print("\nModel architecture saved to 'model_architecture.png'.")

In [ ]:
# Model compilation
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=0.0001,
        verbose=1
    )
]

# Model training
if train_generator.samples > 0:
    print("Starting model training...")
    
    epochs = 20
    
    history = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // batch_size,
        epochs=epochs,
        validation_data=validation_generator,
        validation_steps=validation_generator.samples // batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    print("Model training completed!")
else:
    print("⚠️ Dataset is empty! Model training skipped.")
    history = None

In [ ]:
# Visualize training history
def plot_training_history(history):
    """Visualize training history"""
    if history is None:
        print("Training history not found!")
        return
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy plot
    ax1.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss plot
    ax2.plot(history.history['loss'], label='Training Loss', marker='o')
    ax2.plot(history.history['val_loss'], label='Validation Loss', marker='s')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print best results
    best_train_acc = max(history.history['accuracy'])
    best_val_acc = max(history.history['val_accuracy'])
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    
    print(f"\nTraining Results:")
    print(f"Best training accuracy: {best_train_acc:.4f}")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final training accuracy: {final_train_acc:.4f}")
    print(f"Final validation accuracy: {final_val_acc:.4f}")

# Visualize training history
plot_training_history(history)

## 4. Model Improvement with Transfer Learning

In [ ]:
# Transfer Learning with VGG16 model
def create_transfer_learning_model(input_shape, num_classes):
    """Create transfer learning model using VGG16"""
    # Load pre-trained VGG16 model
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze base model weights
    base_model.trainable = False
    
    # Create new model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Create transfer learning model
print("Creating Transfer Learning model...")
transfer_model, base_model = create_transfer_learning_model(input_shape, num_classes)

# Show model summary
print(f"\nNumber of layers in base model: {len(base_model.layers)}")
print(f"Total layers in transfer model: {len(transfer_model.layers)}")
print(f"Non-trainable parameters: {base_model.count_params()}")

transfer_model.summary()

In [ ]:
# Compile transfer learning model
transfer_model.compile(
    optimizer=Adam(learning_rate=0.0001),  # Lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Transfer learning training
if train_generator.samples > 0:
    print("Training Transfer Learning model...")
    
    # Reset generators
    train_generator.reset()
    validation_generator.reset()
    
    transfer_epochs = 15
    
    transfer_history = transfer_model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // batch_size,
        epochs=transfer_epochs,
        validation_data=validation_generator,
        validation_steps=validation_generator.samples // batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    print("Transfer Learning training completed!")
else:
    print("⚠️ Dataset is empty! Transfer Learning training skipped.")
    transfer_history = None

# Visualize transfer learning results
if transfer_history is not None:
    print("\n=== TRANSFER LEARNING RESULTS ===")
    plot_training_history(transfer_history)

## 5. Model Evaluation and Comparison

In [ ]:
# Model evaluation function
def evaluate_model(model, generator, model_name):
    """Model evaluation function"""
    if generator.samples == 0:
        print(f"{model_name}: Dataset is empty!")
        return
    
    print(f"\n=== {model_name} EVALUATION ===")
    
    # Reset generator
    generator.reset()
    
    # Evaluate model performance
    steps = generator.samples // generator.batch_size
    if steps == 0:
        steps = 1
    
    loss, accuracy = model.evaluate(generator, steps=steps, verbose=0)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    
    # Predictions and true labels
    predictions = model.predict(generator, steps=steps, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Get true labels
    true_classes = generator.classes[:len(predicted_classes)]
    class_labels = list(generator.class_indices.keys())
    
    # Classification report
    print("\nDetailed Report:")
    print(classification_report(true_classes, predicted_classes, target_names=class_labels))
    
    # Confusion Matrix
    cm = confusion_matrix(true_classes, predicted_classes)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.show()
    
    return accuracy

# Evaluate both models
if validation_generator.samples > 0:
    # Simple CNN model evaluation
    if history is not None:
        cnn_accuracy = evaluate_model(model, validation_generator, "Simple CNN")
    
    # Transfer Learning model evaluation
    if transfer_history is not None:
        transfer_accuracy = evaluate_model(transfer_model, validation_generator, "Transfer Learning (VGG16)")
        
        # Model comparison
        if history is not None:
            print(f"\n=== MODEL COMPARISON ===")
            print(f"Simple CNN Accuracy: {cnn_accuracy:.4f}")
            print(f"Transfer Learning Accuracy: {transfer_accuracy:.4f}")
            
            improvement = transfer_accuracy - cnn_accuracy
            print(f"Transfer Learning Improvement: {improvement:.4f} ({improvement*100:.2f}%)")
else:
    print("⚠️ Validation dataset is empty! Model evaluation cannot be performed.")

## 6. Model Saving and Loading

In [ ]:
# Model saving
def save_models():
    """Save trained models"""
    if history is not None:
        model.save('simple_cnn_model.h5')
        print("Simple CNN model saved as 'simple_cnn_model.h5'.")
    
    if transfer_history is not None:
        transfer_model.save('transfer_learning_model.h5')
        print("Transfer Learning model saved as 'transfer_learning_model.h5'.")

# Save models
save_models()

# Model loading example
def load_and_use_model(model_path, test_image_path=None):
    """Load saved model and use it"""
    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}")
        return
    
    # Load model
    loaded_model = keras.models.load_model(model_path)
    print(f"Model loaded: {model_path}")
    
    # If test image is provided, make prediction
    if test_image_path and os.path.exists(test_image_path):
        # Load and preprocess image
        img = Image.open(test_image_path)
        img = img.resize((img_height, img_width))
        img_array = np.array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        # Make prediction
        prediction = loaded_model.predict(img_array)
        predicted_class = classes[np.argmax(prediction)]
        confidence = np.max(prediction)
        
        print(f"Prediction: {predicted_class}")
        print(f"Confidence: {confidence:.4f}")
        
        # Show image
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f'Prediction: {predicted_class} (Confidence: {confidence:.4f})')
        plt.axis('off')
        plt.show()
    
    return loaded_model

print("\nModel loading example:")
print("load_and_use_model('transfer_learning_model.h5', 'test_image.jpg')")

## 7. Results and Evaluation

### Project Summary:
- Developed a 3-class image classification project (cup, pen, keyboard)
- Compared Simple CNN and Transfer Learning (VGG16) models
- Applied data augmentation techniques
- Evaluated model performances with various metrics

### Key Findings:
1. **Transfer Learning superiority**: Pre-trained VGG16 model showed better performance
2. **Importance of data augmentation**: Prevented overfitting in small datasets
3. **Effect of callbacks**: EarlyStopping and LR reduction optimized the model

### Model Performance Improvement Recommendations:

#### 1. **Dataset Improvements:**
- Collect more images for each class (minimum 100-500 examples)
- Improve image quality
- Add examples from different angles and lighting conditions
- Ensure balanced dataset

#### 2. **Model Architecture Improvements:**
- Try modern architectures (ResNet, EfficientNet, Vision Transformer)
- Fine-tuning: Train upper layers of transfer learning model too
- Ensemble methods: Combine predictions from multiple models
- Add attention mechanisms

#### 3. **Hyperparameter Optimization:**
- Learning rate scheduling
- Batch size optimization
- Dropout rate tuning
- Optimizer selection (Adam, AdamW, SGD)

#### 4. **Advanced Data Augmentation:**
- Advanced techniques like Mixup, CutMix
- AutoAugment policies
- Test-time augmentation

### Recommendations for Real-World Application:
- Model quantization for mobile deployment
- Real-time inference optimization
- Edge computing compatibility
- Model monitoring and continuous learning

## 8. Practical Application Guide

### To Prepare Your Own Dataset:

#### Step 1: Data Collection
```bash
# Create folder structure
mkdir -p proje_veri_seti/bardak
mkdir -p proje_veri_seti/kalem  
mkdir -p proje_veri_seti/klavye
```

#### Step 2: Image Properties
- **Format**: JPG, PNG
- **Size**: Minimum 224x224 pixels
- **Quality**: High resolution
- **Diversity**: Different angles, lighting conditions

#### Step 3: Dataset Size
- **Minimum**: 50 images per class
- **Ideal**: 200-500 images per class
- **Production**: 1000+ images per class

### Model Improvement Checklist:

- [ ] **Data quality checked**
- [ ] **Data balancing performed**
- [ ] **Appropriate data augmentation selected**
- [ ] **Transfer learning tried**
- [ ] **Hyperparameter tuning performed**
- [ ] **Cross-validation applied**
- [ ] **Model ensemble tried**
- [ ] **Error analysis performed**

### Preparation for Deployment:

#### Model Export:
```python
# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
```

#### ONNX Format:
```python
# Convert to ONNX format
import tf2onnx
onnx_model = tf2onnx.convert.from_keras(model)
```

### Performance Monitoring:
- **Accuracy**: Overall accuracy
- **Precision/Recall**: Class-based performance
- **F1-Score**: Balanced metric
- **Confusion Matrix**: Error analysis
- **Inference Time**: Prediction speed

## 9. Next Steps

### Immediate Actions:
- [ ] **Prepare your own dataset** (cup, pen, keyboard images)
- [ ] **Collect at least 50 images per class**
- [ ] **Re-run model training**
- [ ] **Document results and prepare report**

### Advanced Developments:
- [ ] **Fine-tuning**: Train transfer learning model in more detail
- [ ] **New architectures**: Try ResNet50, EfficientNet
- [ ] **Data pipeline**: More advanced data loading system
- [ ] **Real-time prediction**: Live prediction with webcam
- [ ] **Web deployment**: Web application with Flask/FastAPI
- [ ] **Mobile app**: Mobile application with TensorFlow Lite

### Research Topics:
- [ ] **Explainable AI**: Explaining model predictions
- [ ] **Few-shot learning**: Learning with few examples
- [ ] **Meta-learning**: Fast adaptation
- [ ] **Adversarial training**: Increasing robustness

### Project Completion Criteria:
- [ ] **Achieve 85%+ test accuracy**
- [ ] **Compare three models** (Simple CNN, Transfer Learning, Custom)
- [ ] **Prepare detailed report**
- [ ] **Create demo application**

---

### 📝 **Note**: To run this notebook with your own dataset:
1. Add your own images to `proje_veri_seti/` folder
2. Run the notebook from start to finish
3. Analyze and report the results

### 🚀 **Good luck!** On your deep learning journey!